# 02 — Funnel Analysis

Recreates and extends the SQL findings using Pandas:
- Overall funnel conversion rates
- Stage-by-stage drop-off
- Channel × stage cross-tabulation
- Time-to-fill distributions
- Sourcing ROI summary
- Recruiter scorecard

Run `01_data_cleaning.ipynb` first — this notebook reads from `candidates_clean`.

In [1]:
import sqlite3
import pandas as pd
import numpy as np

DB_PATH = "../database/recruitment.db"
conn = sqlite3.connect(DB_PATH)

candidates = pd.read_sql("SELECT * FROM candidates_clean", conn)
pipeline   = pd.read_sql("SELECT * FROM pipeline_stages",  conn)
roles      = pd.read_sql("SELECT * FROM roles",            conn)
clients    = pd.read_sql("SELECT * FROM clients",          conn)

pipeline['stage_date'] = pd.to_datetime(pipeline['stage_date'])
roles['date_opened']   = pd.to_datetime(roles['date_opened'])

# Merge channel onto pipeline
pipeline = pipeline.merge(candidates[['candidate_id', 'source_channel']], on='candidate_id', how='left')
# Merge industry onto pipeline via roles → clients
roles_ext = roles.merge(clients[['client_id', 'industry']], on='client_id')
pipeline = pipeline.merge(roles_ext[['role_id', 'industry', 'date_opened']], on='role_id', how='left')

print(f"Pipeline rows: {len(pipeline):,}")

Pipeline rows: 2,987


## 1. Overall funnel conversion

In [2]:
STAGE_ORDER = ['Sourced', 'Screened', 'Interviewed', 'Offered', 'Placed']

stage_counts = (
    pipeline[pipeline['stage'].isin(STAGE_ORDER)]
    .groupby('stage')['candidate_id']
    .nunique()
    .reindex(STAGE_ORDER)
)

sourced_n = stage_counts['Sourced']
funnel = pd.DataFrame({
    'stage': STAGE_ORDER,
    'candidates': stage_counts.values,
    'pct_of_sourced': (stage_counts.values / sourced_n * 100).round(1),
    'step_conversion': [100.0] + [
        round(stage_counts[STAGE_ORDER[i]] / stage_counts[STAGE_ORDER[i-1]] * 100, 1)
        for i in range(1, len(STAGE_ORDER))
    ]
})
print(funnel.to_string(index=False))

      stage  candidates  pct_of_sourced  step_conversion
    Sourced        1000           100.0            100.0
   Screened         578            57.8             57.8
Interviewed         288            28.8             49.8
    Offered         121            12.1             42.0
     Placed          94             9.4             77.7


## 2. Conversion rates by source channel

In [3]:
channel_funnel = (
    pipeline[pipeline['stage'].isin(STAGE_ORDER)]
    .groupby(['source_channel', 'stage'])['candidate_id']
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=STAGE_ORDER, fill_value=0)
)

channel_funnel['placement_rate_pct'] = (
    channel_funnel['Placed'] / channel_funnel['Sourced'] * 100
).round(1)

print(channel_funnel.sort_values('placement_rate_pct', ascending=False).to_string())

stage            Sourced  Screened  Interviewed  Offered  Placed  placement_rate_pct
source_channel                                                                      
Job Board            208       120           76       33      26                12.5
Referral             186        98           47       23      19                10.2
Agency Database      192       112           45       21      17                 8.9
Cold Outreach        199       112           57       22      16                 8.0
LinkedIn             215       136           63       22      16                 7.4


## 3. Time-to-fill analysis

In [4]:
placed = pipeline[pipeline['stage'] == 'Placed'].copy()

first_placement = (
    placed.sort_values('stage_date')
    .groupby('role_id')
    .agg(
        industry=('industry', 'first'),
        date_placed=('stage_date', 'first'),
        date_opened=('date_opened', 'first')
    )
    .reset_index()
)
first_placement['days_to_fill'] = (
    (first_placement['date_placed'] - first_placement['date_opened']).dt.days
    .clip(lower=1)
)

print(f"Roles filled: {len(first_placement)}")
print(f"\nTime-to-fill summary (days):")
print(first_placement['days_to_fill'].describe().round(1).to_string())


Roles filled: 32

Time-to-fill summary (days):
count     32.0
mean      54.5
std       35.4
min        1.0
25%       30.0
50%       45.5
75%       67.8
max      188.0


In [5]:
# By industry
ttf_by_industry = (
    first_placement.groupby('industry')['days_to_fill']
    .agg(['count', 'mean', 'median'])
    .round(1)
    .rename(columns={'count': 'roles_filled', 'mean': 'avg_days', 'median': 'median_days'})
    .sort_values('avg_days', ascending=False)
)
print(ttf_by_industry.to_string())

               roles_filled  avg_days  median_days
industry                                          
Legal                     8      63.1         59.0
Finance                  18      55.1         39.5
Manufacturing             5      44.0         35.0
Healthcare                1      27.0         27.0


In [6]:
# Monthly trend
first_placement['month'] = first_placement['date_placed'].dt.to_period('M')
monthly_ttf = (
    first_placement.groupby('month')['days_to_fill']
    .agg(['count', 'mean'])
    .round(1)
    .rename(columns={'count': 'placements', 'mean': 'avg_days_to_fill'})
)
print(monthly_ttf.to_string())

         placements  avg_days_to_fill
month                                
2024-02           2              15.5
2024-03           3              35.3
2024-04           1              29.0
2024-05           2              32.0
2024-06           2              66.5
2024-07           2              52.5
2024-08           1             100.0
2024-09           2             112.5
2024-10           3              42.7
2024-11           2              53.0
2024-12           3              64.0
2025-01           1              59.0
2025-02           4              53.5
2025-04           1              56.0
2025-05           1              59.0
2025-06           2              68.0


## 4. Sourcing ROI

In [7]:
# ASSUMPTION: nominal cost-per-candidate-sourced by channel (illustrative figures)
CHANNEL_COST = {
    'LinkedIn':        45,
    'Referral':        10,
    'Job Board':       60,
    'Agency Database': 15,
    'Cold Outreach':   25,
}

roi = channel_funnel[['Sourced', 'Placed']].copy()
roi['cost_per_sourced_gbp']  = roi.index.map(CHANNEL_COST)
roi['total_spend_gbp']       = roi['Sourced'] * roi['cost_per_sourced_gbp']
roi['cost_per_hire_gbp']     = (roi['total_spend_gbp'] / roi['Placed']).round(0)
roi['placement_rate_pct']    = (roi['Placed'] / roi['Sourced'] * 100).round(1)

print(roi.sort_values('cost_per_hire_gbp').to_string())

stage            Sourced  Placed  cost_per_sourced_gbp  total_spend_gbp  cost_per_hire_gbp  placement_rate_pct
source_channel                                                                                                
Referral             186      19                    10             1860               98.0                10.2
Agency Database      192      17                    15             2880              169.0                 8.9
Cold Outreach        199      16                    25             4975              311.0                 8.0
Job Board            208      26                    60            12480              480.0                12.5
LinkedIn             215      16                    45             9675              605.0                 7.4


## 5. Recruiter scorecard

In [8]:
r_sourced = (
    pipeline[pipeline['stage'] == 'Sourced']
    .groupby('recruiter')['candidate_id'].nunique().rename('sourced')
)
r_placed = (
    pipeline[pipeline['stage'] == 'Placed']
    .groupby('recruiter')['candidate_id'].nunique().rename('placed')
)
sc = pd.concat([r_sourced, r_placed], axis=1).fillna(0)
sc['placement_rate_pct'] = (sc['placed'] / sc['sourced'] * 100).round(1)

recruiter_ttf = (
    placed
    .assign(days_to_fill=lambda df: (df['stage_date'] - df['date_opened']).dt.days)
    .groupby('recruiter')['days_to_fill'].mean().round(1)
    .rename('avg_days_to_fill')
)
scorecard = sc.join(recruiter_ttf)
print(scorecard.sort_values('placed', ascending=False).to_string())


                 sourced  placed  placement_rate_pct  avg_days_to_fill
recruiter                                                             
Marcus Chen          267      29                10.9             122.9
Sarah Kelly          221      25                11.3             142.7
Dave O'Sullivan      273      21                 7.7             125.7
Priya Nair           239      19                 7.9              83.9


In [9]:
conn.close()
print("Analysis complete.")

Analysis complete.
